In [ ]:
%pip install tensorflow scikit-learn pandas pandas-market-calendars

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import Input, Conv1D, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from pandas.tseries.offsets import BDay

tf.config.threading.set_inter_op_parallelism_threads(2)
tf.config.threading.set_intra_op_parallelism_threads(2)


: 

In [ ]:
df = pd.read_csv("NSE_1001_TO_1050_start_to_15082020.csv")

# ---- AUTO DETECT COMPANY COLUMN ----
possible_cols = ["Symbol", "SYMBOL", "symbol", "Company", "company"]

company_col = None
for c in possible_cols:
    if c in df.columns:
        company_col = c
        break

if company_col is None:
    raise Exception("❌ No company column found (Symbol/Company missing)")

print("Using company column:", company_col)

df['Date'] = pd.to_datetime(df['Date'])
df.sort_values(['Date'], inplace=True)


In [ ]:
def add_indicators(df):

    df['Return'] = df['Close'].pct_change()

    delta = df['Close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / (avg_loss + 1e-9)
    df['RSI'] = 100 - (100/(1+rs))

    df['EMA'] = df['Close'].ewm(span=14, adjust=False).mean()

    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26

    df.dropna(inplace=True)

    return df


In [ ]:
SEQ_LEN = 60

def create_sequences(data):
    X, y = [], []

    for i in range(SEQ_LEN, len(data)):
        X.append(data[i-SEQ_LEN:i])
        y.append(data[i,0])

    return np.array(X), np.array(y)


In [ ]:
def build_model(n_features):

    inp = Input(shape=(SEQ_LEN, n_features))

    x = Conv1D(32, 3, activation="relu")(inp)
    x = BatchNormalization()(x)

    x = LSTM(48)(x)
    x = Dropout(0.2)(x)

    out = Dense(1)(x)

    model = Model(inp, out)

    model.compile(
        optimizer=Adam(0.001),
        loss="mse"
    )

    return model


In [ ]:
@tf.function(reduce_retracing=True)
def predict_step(model, x):
    return model(x, training=False)


In [ ]:
results = []

companies = df[company_col].unique()

for company in companies:

    print("Processing:", company)

    data = df[df[company_col] == company].copy()

    if len(data) < 200:
        continue

    data = add_indicators(data)

    features = data[['Close','RSI','MACD','EMA','Return']].values

    # ✅ scaler per company (VERY IMPORTANT)
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(features)

    X, y = create_sequences(scaled)

    split = int(len(X)*0.8)
    X_train, y_train = X[:split], y[:split]

    model = build_model(X.shape[2])

    model.fit(
        X_train, y_train,
        epochs=8,
        batch_size=32,
        verbose=0
    )

    # ---------- Prediction ----------
    last_seq = scaled[-SEQ_LEN:]
    last_seq = np.expand_dims(last_seq, axis=0)

    pred_scaled = predict_step(model, last_seq).numpy()

    temp = np.zeros((1, features.shape[1]))
    temp[0,0] = pred_scaled[0,0]

    pred_price = scaler.inverse_transform(temp)[0,0]
    current_price = data['Close'].iloc[-1]

    change_pct = ((pred_price-current_price)/current_price)*100
    direction = "UP 📈" if change_pct > 0 else "DOWN 📉"

    last_date = data['Date'].iloc[-1]
    prediction_date = last_date + BDay(1)

    results.append([
        company,
        last_date.date(),
        prediction_date.date(),
        round(current_price,2),
        round(pred_price,2),
        direction,
        round(change_pct,2)
    ])


In [ ]:
results_df = pd.DataFrame(results, columns=[
    "Company",
    "Last Data Date",
    "Prediction Date",
    "Current Price (₹)",
    "Predicted Next Day Price (₹)",
    "Direction",
    "Change %"
])

print("\n========== FINAL PREDICTIONS ==========\n")
print(results_df.to_string(index=False))

# Save to CSV
results_df.to_csv("stock_predictions_lstm.csv", index=False)
print("\n✅ Results saved to 'stock_predictions_lstm.csv'")